In [40]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

class Neo4jHttpGraph:
    def __init__(self, query_url=None, username=None, password=None):
        self.query_url = query_url or f"https://{os.getenv('AURA_INSTANCEID')}.databases.neo4j.io/db/{os.getenv('NEO4J_DATABASE')}/query/v2"
        self.username = username or os.getenv("AURA_INSTANCEID")
        self.password = password or os.getenv("NEO4J_PASSWORD")
        self.auth = (self.username, self.password)
        self.schema = ""
        self._enhanced_schema = False
        
        # Connection test
        self.query("RETURN 1 AS test")
        print("Connected to Neo4j via HTTP Query API successfully!")
        self.refresh_schema()

    def query(self, query: str, params: dict = None):
        payload = {"statement": query}
        if params:
            payload["parameters"] = params
        resp = requests.post(self.query_url, json=payload, auth=self.auth)
        resp.raise_for_status()
        data = resp.json().get("data", {})
        fields = data.get("fields", [])
        values = data.get("values", [])
        return [dict(zip(fields, row)) for row in values]

    @property
    def get_structured_schema(self) -> dict:
        """Structured schema required by GraphCypherQAChain."""
        node_props_query = """
        CALL db.schema.nodeTypeProperties()
        YIELD nodeType, propertyName, propertyTypes
        WHERE NOT nodeType CONTAINS '__Entity__'
        RETURN nodeType, collect({property: propertyName, type: propertyTypes[0]}) AS properties
        """
        rel_query = """
        CALL db.schema.visualization()
        YIELD nodes, relationships
        RETURN [r IN relationships | {
            start: [l IN labels(startNode(r)) WHERE l <> '__Entity__'][0],
            type: type(r),
            end: [l IN labels(endNode(r)) WHERE l <> '__Entity__'][0]
        }] AS rels
        """
        node_props = {}
        try:
            for row in self.query(node_props_query):
                lbl = row["nodeType"].replace(":", "").replace("`", "")
                node_props[lbl] = row.get("properties", [])
        except Exception:
            pass

        rels = []
        try:
            res = self.query(rel_query)
            if res and res[0].get("rels"):
                rels = [r for r in res[0]["rels"] if r.get("start") and r.get("end")]
        except Exception:
            pass

        return {
            "node_props": node_props,
            "relationships": rels,
            "metadata": {}
        }

    def refresh_schema(self):
        """Builds schema string matching the standard LangChain Neo4jGraph output."""
        try:
            node_props_query = """
            CALL db.schema.nodeTypeProperties()
            YIELD nodeType, propertyName, propertyTypes
            WHERE NOT nodeType CONTAINS '__Entity__'
            RETURN nodeType, collect({property: propertyName, type: propertyTypes[0]}) AS properties
            """
            node_res = self.query(node_props_query)
            
            node_str = "Node properties:\n"
            for row in node_res:
                lbl = row["nodeType"].replace(":", "").replace("`", "")
                props = ", ".join([f"{p['property']}: {p['type']}" for p in row.get("properties", []) if p.get('property')])
                node_str += f"{lbl} {{{props}}}\n"

            rel_query = """
            CALL db.schema.visualization()
            YIELD nodes, relationships
            RETURN [r IN relationships | {
                start: [l IN labels(startNode(r)) WHERE l <> '__Entity__'][0],
                type: type(r),
                end: [l IN labels(endNode(r)) WHERE l <> '__Entity__'][0]
            }] AS rels
            """
            rel_res = self.query(rel_query)
            
            rel_str = "Relationship properties:\n\nThe relationships:\n"
            if rel_res and rel_res[0].get("rels"):
                for r in rel_res[0]["rels"]:
                    if r.get("start") and r.get("end"):
                        rel_str += f"(:{r['start']})-[:{r['type']}]->(:{r['end']})\n"

            self.schema = f"{node_str}{rel_str}".strip()
        except Exception:
            pass

    def get_schema(self) -> str:
        if not self.schema:
            self.refresh_schema()
        return self.schema

    def add_graph_documents(self, graph_documents, baseEntityLabel=False, include_source=True):
        """Loads GraphDocuments, creates Document nodes and MENTIONS relationships."""
        for doc in graph_documents:
            # 1. Add Source Document node
            doc_id = None
            if include_source and getattr(doc, "source", None):
                doc_text = doc.source.page_content
                doc_id = str(abs(hash(doc_text)))
                self.query("""
                MERGE (d:Document {id: $id})
                SET d.text = $text
                """, params={"id": doc_id, "text": doc_text})

            # 2. Add Nodes & Link with MENTIONS
            for node in doc.nodes:
                labels = [node.type]
                if baseEntityLabel:
                    labels.append("__Entity__")
                label_str = ":".join(f"`{lbl}`" for lbl in labels if lbl)
                
                query = f"""
                MERGE (n:{label_str} {{id: $id}})
                SET n += $properties
                """
                self.query(query, params={"id": node.id, "properties": node.properties or {}})

                if doc_id:
                    self.query(f"""
                    MATCH (d:Document {{id: $doc_id}})
                    MATCH (n:`{node.type}` {{id: $node_id}})
                    MERGE (d)-[:MENTIONS]->(n)
                    """, params={"doc_id": doc_id, "node_id": node.id})

            # 3. Add Graph Relationships
            for rel in doc.relationships:
                query = f"""
                MATCH (s:`{rel.source.type}` {{id: $source_id}})
                MATCH (t:`{rel.target.type}` {{id: $target_id}})
                MERGE (s)-[r:`{rel.type}`]->(t)
                SET r += $properties
                """
                self.query(query, params={
                    "source_id": rel.source.id,
                    "target_id": rel.target.id,
                    "properties": rel.properties or {}
                })

# Initialize instance
graph = Neo4jHttpGraph(
    query_url="https://b4b9f51e.databases.neo4j.io/db/b4b9f51e/query/v2",
    username=os.getenv("AURA_INSTANCEID"), 
    password=os.getenv("NEO4J_PASSWORD")
)


Connected to Neo4j via HTTP Query API successfully!


In [41]:
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    groq_api_key=groq_api_key,
    reasoning_effort="low",
    model_kwargs={
        "parallel_tool_calls": False
    }
)

In [42]:
from langchain_core.documents import Document

movie_texts = [
    "Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.",
    "Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.",
    "Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008."
]

documents = [Document(page_content=text) for text in movie_texts]

documents


[Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.'),
 Document(metadata={}, page_content='Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.'),
 Document(metadata={}, page_content='Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.')]

In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:", chunk.page_content)

Chunk 1: Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.
Chunk 2: Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.
Chunk 3: Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.


In [44]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Movie", "Genre"],
    allowed_relationships=["DIRECTED", "ACTED_IN", "IN_GENRE"]
)

graph_documents = llm_transformer.convert_to_graph_documents(chunks)


In [45]:
graph_documents

[GraphDocument(nodes=[Node(id='Christopher Nolan', type='Person', properties={}), Node(id='Inception', type='Movie', properties={}), Node(id='Leonardo Dicaprio', type='Person', properties={}), Node(id='Science Fiction', type='Genre', properties={})], relationships=[Relationship(source=Node(id='Christopher Nolan', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='DIRECTED', properties={}), Relationship(source=Node(id='Leonardo Dicaprio', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='ACTED_IN', properties={}), Relationship(source=Node(id='Inception', type='Movie', properties={}), target=Node(id='Science Fiction', type='Genre', properties={}), type='IN_GENRE', properties={})], source=Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.')),
 GraphDocument(nodes=[Node(id='Christopher No

In [46]:
for i, graph_doc in enumerate(graph_documents):
    print(f"\n--- Graph Document {i+1} ---")
    print("Nodes:")
    for node in graph_doc.nodes:
        print(node)
    print("Relationships:")
    for rel in graph_doc.relationships:
        print(rel)


--- Graph Document 1 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Inception' type='Movie' properties={}
id='Leonardo Dicaprio' type='Person' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='DIRECTED' properties={}
source=Node(id='Leonardo Dicaprio', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='ACTED_IN' properties={}
source=Node(id='Inception', type='Movie', properties={}) target=Node(id='Science Fiction', type='Genre', properties={}) type='IN_GENRE' properties={}

--- Graph Document 2 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Interstellar' type='Movie' properties={}
id='Matthew Mcconaughey' type='Person' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properti

In [47]:
graph.query("MATCH (n) DETACH DELETE n")


[]

In [48]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=False,
    include_source=True
)




In [49]:
graph.refresh_schema()
print(graph.schema)


Node properties:
Movie {id: STRING NOT NULL}
Genre {id: STRING NOT NULL}
Document {id: STRING NOT NULL, text: STRING NOT NULL}
Person {id: STRING NOT NULL}
Relationship properties:

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Document)-[:MENTIONS]->(:Genre)
(:Document)-[:MENTIONS]->(:Movie)
(:Document)-[:MENTIONS]->(:Person)
(:Person)-[:DIRECTED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


In [50]:
result = graph.query("""
MATCH (director:Person)-[:DIRECTED]->(movie:Movie)
OPTIONAL MATCH (actor:Person)-[:ACTED_IN]->(movie)
RETURN director.id AS director, movie.id AS movie, collect(actor.id) AS actors
""")

result

[{'director': 'Christopher Nolan',
  'movie': 'Inception',
  'actors': ['Leonardo Dicaprio']},
 {'director': 'Christopher Nolan',
  'movie': 'Interstellar',
  'actors': ['Matthew Mcconaughey']},
 {'director': 'Christopher Nolan',
  'movie': 'The Dark Knight',
  'actors': ['Christian Bale']}]

In [51]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)

In [52]:
response = chain.invoke({
    "query": "Which movies did Christopher Nolan direct?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie)
WHERE p.id = "Christopher Nolan"
RETURN m
Full Context:
[{'m': {'elementId': '4:3d860959-d475-4338-8fac-403aa2b72739:2', 'labels': ['Movie'], 'properties': {'id': 'Inception'}}}, {'m': {'elementId': '4:3d860959-d475-4338-8fac-403aa2b72739:6', 'labels': ['Movie'], 'properties': {'id': 'Interstellar'}}}, {'m': {'elementId': '4:3d860959-d475-4338-8fac-403aa2b72739:9', 'labels': ['Movie'], 'properties': {'id': 'The Dark Knight'}}}]

> Finished chain.
Christopher Nolan directed Inception, Interstellar, and The Dark Knight.


In [53]:
response = chain.invoke({
    "query": "Which movie did Christopher Nolan direct in 2010 and who acted in it?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (director:Person {id: "Christopher Nolan"})-[:DIRECTED]->(m:Movie)
WHERE m.id CONTAINS "2010"
MATCH (actor:Person)-[:ACTED_IN]->(m)
RETURN m, actor
Full Context:
[]

> Finished chain.
I don't know the answer.
